# 13 - Train Multi-Head n-step DQN Offline

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, but
trains **several Q heads**, each with its own n-step backup, instead of one one-step
head:

1. Load previously collected `Datastore` streams from the Hub.
1. Build a `DataLoader` that fills one batch with whole episodes up to `TOKEN_BUDGET`
   packed tokens.
1. Assemble a `Model` from a backbone and **one `MLP` per horizon**.
1. Call `DqnObjective` once per head (`nstep_gate(n=...)`) and **sum** the losses. Save
   with `push_model_to_hub`.

An n-step target is the discounted sum of the next `n` rewards plus a delayed
state-value bootstrap at `s_{t+n}` (the backup action scores select an action and the
backup value scores evaluate it; positive temperature uses their expected value under
the action-score policy plus entropy):

`G_t^{(n)} = r_{t+1} + γ_{t+1} r_{t+2} + … + (∏ γ) V(s_{t+n})`

`n=1` is ordinary one-step DQN. A longer `n` puts more of the return on the observed
rewards and less on the delayed network. A zero discount at outer task termination or
truncation ends the return before it can leave the sampled group.
`environment_episode_ranges` selects complete outer tasks from the event log. Unfinished
tails remain in storage. The objective raises if a training target needs a value beyond
its sampled group; zero discount at outer termination or truncation ends the return.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in
`15_inference.ipynb`.


In [ ]:
import json
from copy import deepcopy
from functools import partial

import torch
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    Augmenter,
    DataLoader,
    EnvironmentProtocol,
    Tokenizer,
    compose,
    environment_episode_ranges,
    load_stores_from_hub,
)
from mouse_core.models import (
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    prepare_decode,
    push_model_to_hub,
)
from mouse_core.objectives import DqnObjective, boundary_discount, nstep_gate

# Repositories
# Hugging Face dataset repo for load_stores_from_hub
DATASET_ID = "mouse-example-frozenlake-dataset"
# Hugging Face model repo for push_model_to_hub
MODEL_ID = "mouse-example-model-nstep-offline"

# Environment
# number of discrete actions predicted by each head
MAX_ACTIONS = 4
# vocabulary size for discrete observations
MAX_OBS_DISCRETE = 64
# max steps per episode (matches collect)
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task (matches collect)
REPEAT_COUNT = 20

# Model
# HF checkpoint for Tokenizer, Embedding, and Backbone
PRETRAINED = "Qwen/Qwen3-0.6B"
HEAD_LAYERS = 1
# final prediction linear's initialization
HEAD_SCALE = 0.1

# Training
# packed tokens; whole episodes are added until the next one would not fit
TOKEN_BUDGET = 16384
# outer train cycles (print cadence)
NUM_CYCLES = 2
# optimizer updates per cycle (passed to run_train)
TRAIN_STEPS = 50

# Objective
N_STEPS = (1, 3, 5)  # short, medium, and long backup horizons

# Optimizer learning rates
# token embeddings and numeric Fourier features
LR_EMBEDDING = 1e-05
# backbone
LR_BACKBONE = 1e-05
# prediction heads and their input RMSNorms
LR_HEADS = 1e-05

# Target interpolation rates
# delayed embedding
POLYAK_TAU_EMBEDDING = 0.1
# delayed backbone
POLYAK_TAU_BACKBONE = 0.1
# delayed value head
POLYAK_TAU_VALUE = 0.001

# Runtime
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Payload names are selected explicitly; the underscore is only a convention.
environment_episode_ranges = partial(
    environment_episode_ranges,
    protocol=EnvironmentProtocol(
        episode_start="_episode_start",
        terminated="_terminated",
        truncated="_truncated",
    ),
)

## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved
`Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)

## Data Pipeline

Use the event pipeline described in [02 — Offline DQN](02_train_offline_dqn.ipynb):
`source events → sparse decisions → augmenter → tokenizer → pack → embedding`.

Compose `train_transform = compose(stages=(augmenter.events, tokenizer.encode))` and
pass `train_transform` to the loader. The augmenter reads event payloads directly. Each
sampled outer task gets a separate `group_id`, retaining its repeated lake episodes
together while isolating attention and objective transitions from other tasks.
`TOKEN_BUDGET` counts packed tokens, including episode prompts; the loader keeps whole
samples and rejects an oversized first sample.

[Example 15](15_inference.ipynb) demonstrates cached inference with example 02's single
Q head. To evaluate this checkpoint, reconstruct this notebook's `q_heads` container and
matching encoder, load its saved tokenizer without augmentation, and choose which return
horizon's head will select actions.


In [ ]:
def episode_separator(ctx):
    ended = ctx.get("inner_terminated", False) or ctx.get("inner_truncated", False)
    return "\n\n" if ended else ""


# Pipeline order: augmenter → tokenizer
episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\n"
)

augmenter = Augmenter(
    seed_field="episode_index",
    fields=[
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "action",
            "input_vector_field": "_q_star",
            "vocab_size": MAX_ACTIONS,
        },
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
        },
    ],
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "episode {field+1}\n",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "lookup": ["a", "b", "c", "d"],
            "format": "{field}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ",r={field:g}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "format": episode_separator,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
    ],
)

train_transform = compose(stages=(augmenter.events, tokenizer.encode))


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=environment_episode_ranges,
    transform=train_transform,
    prefetch=4,
    num_workers=4,
)

## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference uses
the same encoder and returns cache and readout-mask metadata. A `ModuleDict` called
`modules` holds references to the online components for checkpointing and mode changes;
it is not called for predictions.

The delayed encoder is a frozen copy. Separate backup value/action heads track the
online Q head(s) with independent Polyak rates. Only online parameters enter the
optimizer.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, train_kernel, decode_kernel, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            train_kernel=train_kernel,
            decode_kernel=decode_kernel,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_token_mask=tensors["head_token_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class QHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        train_kernel="flex",
        decode_kernel="flex",
        dtype=torch.float32,
    )
    .train()
    .to(device)
)

head_keys = tuple(f"action_value_{n}" for n in N_STEPS)
q_heads = (
    nn.ModuleDict(
        {
            key: QHead(
                hidden_dim=encoder.backbone.hidden_dim,
                num_actions=MAX_ACTIONS,
                num_layers=HEAD_LAYERS,
                scale=HEAD_SCALE,
            )
            for key in head_keys
        }
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_heads": q_heads})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_heads = deepcopy(q_heads).requires_grad_(False).eval()

print(encoder)
print(q_heads)

## Train

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core
abstractions do most of the work:

`gate=nstep_gate(n=...)` is the n-step return. Along a run the target of horizon `n` is

`G_i^{(n)} = r_i + γ_i r_{i+1} + … + (∏_{k=0}^{n-1} γ_{i+k}) V(s_{i+n})`,

where `V` evaluates that horizon's backup value head at the action chosen by its backup
action head. Positive temperature uses the action head's Boltzmann policy and adds its
entropy bonus. `n=1` is `r + γ V`. `environment_episode_ranges` selects complete outer
tasks from the event log. Unfinished tails remain in storage. The objective raises if a
training target needs a value beyond its sampled group; zero discount at outer
termination or truncation ends the return.

Rewards and values pass through unchanged (`reward=None`, `value=None`).
`boundary_discount` applies each transition's gamma to later rewards and the bootstrap.
Outer task termination or truncation sets it to zero; inner episode boundaries keep the
return connected across repeats.

The loader returns `inputs` (tokens, embedding features, sequence IDs, and
`head_token_mask`) and `objective_data` (training labels and step mappings).
`objective_data["head_step_ids"][p]` tells the loss which step owns prediction row `p`,
including steps with zero or multiple predictions. Pass this existing mapping to the
objective; no separate prediction-count column is needed. An absent action is
represented by `NaN` in the packed labels.


In [ ]:
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": q_heads.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objectives = [
    DqnObjective(
        gate=nstep_gate(n=n),
        reward=None,
        value=None,
        discount=boundary_discount(
            gamma_step=1.0,
            gamma_terminated=0.0,
            gamma_truncated=0.0,
        ),
        temperature=0.0,
    )
    for n in N_STEPS
]


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    modules.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        group_id = objective_data["step_sequence_ids"]
        out = encoder(inputs)
        with torch.no_grad():
            delayed_z = delayed_encoder(inputs)["z"]
            backup_predictions = {
                **{
                    f"value_{key}": backup_value_heads[key](delayed_z)
                    for key in head_keys
                },
                **{f"action_{key}": q_heads[key](delayed_z) for key in head_keys},
            }
        batch = objective_data.to(device)
        loss = None
        metrics = {}
        for key, objective in zip(head_keys, objectives):
            part, part_metrics = objective(
                objective_data=batch,
                group_id=group_id.to(device),
                head_step_ids=objective_data["head_step_ids"],
                predictions=q_heads[key](out["z"]),
                backup_value_predictions=backup_predictions[f"value_{key}"],
                backup_action_predictions=backup_predictions[f"action_{key}"],
            )
            part_loss = torch.stack(tuple(part.values())).sum()
            loss = part_loss if loss is None else loss + part_loss
            metrics.update(
                {f"{key}_{name}": value for name, value in part_metrics.items()}
            )
        assert loss is not None
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak_update(
            target=delayed_encoder.embedding,
            source=encoder.embedding,
            tau=POLYAK_TAU_EMBEDDING,
        )
        polyak_update(
            target=delayed_encoder.backbone,
            source=encoder.backbone,
            tau=POLYAK_TAU_BACKBONE,
        )
        for key in head_keys:
            polyak_update(
                target=backup_value_heads[key],
                source=q_heads[key],
                tau=POLYAK_TAU_VALUE,
            )
    assert loss is not None
    return (loss, metrics)

## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the
checkpoint later in `15_inference.ipynb` (`get_action` reads `action_value_5`).


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(num_steps=TRAIN_STEPS)
    parts = "  ".join(
        (f"n{n}={metrics[f'action_value_{n}_action_value']:.4f}" for n in N_STEPS)
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  {parts}")

loader.close()

## Save To The Hub

The checkpoint contains the encoder and separate head(s) in `modules`. The model weights
and tokenizer packing specification share `MODEL_ID`. Reconstruct the same component
structure before loading. Checkpoints from the old architecture with a projection are
not compatible with this example.


In [ ]:
modules.eval().to("cpu")
model_url = push_model_to_hub(
    model=modules,
    tokenizer=tokenizer,
    repo_id=MODEL_ID,
    private=False,
    clear=True,
)
print(f"Pushed encoder, heads, and tokenizer to {model_url}")